#### **Likelihood-Informed Subspaces INTERACTIVE EXPLORER**


In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import multivariate_normal
import sys
from pathlib import Path

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "boed" / "__init__.py").is_file():
            return p
        if (p / "pyBOED" / "boed" / "__init__.py").is_file():
            return p / "pyBOED"
    return cwd

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "boed" or name.startswith("boed."):
        del sys.modules[name]

repo_root = PROJECT_ROOT
print(f"Using PROJECT_ROOT={PROJECT_ROOT}")
from boed.reduction.methods import LikelihoodInformedSubspaces, DataAveragedLIS

Using PROJECT_ROOT=/home/mdoumbou/Documents/Biblio_thèse/pyBOED


In [2]:
# ============================================================================
# WIDGETS
# ============================================================================

# Paramètres du problème
d_param_slider = widgets.IntSlider(
    value=10,
    min=5,
    max=50,
    step=5,
    description='Param dim (d):',
    continuous_update=False
)

m_obs_slider = widgets.IntSlider(
    value=5,
    min=2,
    max=20,
    step=1,
    description='Obs dim (m):',
    continuous_update=False
)

# Forward model type
forward_model_dropdown = widgets.Dropdown(
    options=[
        ('Linear: G(x) = Ax', 'linear'),
        ('Polynomial: G(x) = Ax + x²', 'poly'),
        ('Nonlinear: G(x) = A sin(x)', 'nonlinear')
    ],
    value='linear',
    description='Forward model:'
)

# Observation noise
obs_noise_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=2.0,
    step=0.1,
    description='Obs noise σ:',
    continuous_update=False
)

# Prior
prior_type_dropdown = widgets.Dropdown(
    options=[
        ('Standard N(0,I)', 'standard'),
        ('Correlated', 'correlated')
    ],
    value='standard',
    description='Prior:'
)

# Paramètres LIS
n_samples_post_slider = widgets.IntSlider(
    value=500,
    min=100,
    max=2000,
    step=100,
    description='# Posterior:',
    continuous_update=False
)

n_samples_prior_slider = widgets.IntSlider(
    value=500,
    min=100,
    max=2000,
    step=100,
    description='# Prior (DA):',
    continuous_update=False
)

kl_threshold_slider = widgets.FloatSlider(
    value=0.01,
    min=0.001,
    max=0.1,
    step=0.005,
    description='D_KL thresh:',
    continuous_update=False,
    readout_format='.4f'
)

# Méthode
method_toggle = widgets.ToggleButtons(
    options=['LIS', 'DA-LIS', 'Both'],
    value='Both',
    description='Method:',
    button_style='info'
)

compute_button_lis = widgets.Button(
    description='Compute LIS',
    button_style='success',
    icon='play'
)

output_lis = widgets.Output()

In [3]:
# ============================================================================
# FONCTIONS
# ============================================================================

def create_forward_model(d, m, model_type='linear'):
    """Crée le forward model."""
    
    np.random.seed(42)
    
    if model_type == 'linear':
        # G(x) = A x
        A = np.random.randn(m, d)
        # Amplifier les premières colonnes
        A[:, :2] *= 3
        
        def forward(x):
            if x.ndim == 1:
                return A @ x, A.T
            else:
                return x @ A.T, np.tile(A.T, (x.shape[0], 1, 1)).transpose(0, 2, 1)
    
    elif model_type == 'poly':
        # G(x) = A x + diag(x²)[:m]
        A = np.random.randn(m, d)
        A[:, :2] *= 3
        
        def forward(x):
            if x.ndim == 1:
                G = A @ x + x[:m]**2
                grad_G = A.T + 2 * np.diag(x[:m] if len(x) >= m else np.pad(x, (0, m-len(x))))
                return G, grad_G
            else:
                G = x @ A.T
                for i in range(len(x)):
                    G[i] += x[i, :m]**2
                # Gradients approximatifs
                grads = np.tile(A.T, (x.shape[0], 1, 1)).transpose(0, 2, 1)
                return G, grads
    
    elif model_type == 'nonlinear':
        # G(x) = A sin(x)
        A = np.random.randn(m, d)
        A[:, :2] *= 3
        
        def forward(x):
            if x.ndim == 1:
                G = A @ np.sin(x)
                grad_G = A.T * np.cos(x)
                return G, grad_G
            else:
                G = np.sin(x) @ A.T
                grads = np.zeros((x.shape[0], m, d))
                for i in range(x.shape[0]):
                    grads[i] = A.T * np.cos(x[i])
                return G, grads
    
    return forward

In [4]:

def create_prior(d, prior_type='standard'):
    """Crée la distribution a priori."""
    
    if prior_type == 'standard':
        mean = np.zeros(d)
        cov = np.eye(d)
    
    elif prior_type == 'correlated':
        mean = np.zeros(d)
        cov = np.zeros((d, d))
        for i in range(d):
            for j in range(d):
                cov[i, j] = np.exp(-abs(i - j) / 3.0)
    
    return mean, cov


In [5]:
def compute_posterior_gaussian(y_obs, forward_func, prior_mean, prior_cov, obs_cov):
    """Calcule le posterior pour problème linéaire Gaussien."""
    
    # Pour G(x) = A x linéaire
    # Posterior = N(μ_post, Σ_post)
    # Σ_post = (A^T Σ_obs^{-1} A + Σ_pr^{-1})^{-1}
    # μ_post = Σ_post A^T Σ_obs^{-1} y
    
    # On suppose forward_func retourne (G, Jacobian)
    _, A_T = forward_func(prior_mean)
    
    if A_T.ndim == 3:
        A = A_T[0].T  # Prendre première Jacobienne si batch
    else:
        A = A_T.T
    
    obs_precision = np.linalg.inv(obs_cov)
    prior_precision = np.linalg.inv(prior_cov)
    
    post_precision = A.T @ obs_precision @ A + prior_precision
    post_cov = np.linalg.inv(post_precision)
    post_mean = post_cov @ A.T @ obs_precision @ y_obs
    
    return post_mean, post_cov

In [6]:
def compute_lis_interactive(button=None):
    """Calcule LIS et visualise."""
    
    with output_lis:
        clear_output(wait=True)
        
        print("⏳ Construction du problème inverse...")
        
        d = d_param_slider.value
        m = m_obs_slider.value
        
        # Forward model
        forward_func = create_forward_model(d, m, forward_model_dropdown.value)
        
        # Prior
        prior_mean, prior_cov = create_prior(d, prior_type_dropdown.value)
        
        # Observation covariance
        obs_cov = (obs_noise_slider.value ** 2) * np.eye(m)
        
        print(f"  Dimension paramètre   : {d}")
        print(f"  Dimension observation : {m}")
        print(f"  Forward model         : {forward_model_dropdown.label}")
        print(f"  Prior                 : {prior_type_dropdown.label}")
        
        # Générer vraie valeur et données
        np.random.seed(42)
        x_true = np.random.randn(d)
        x_true[:2] = [2.0, -1.5]  # Valeurs significatives
        
        y_true, _ = forward_func(x_true)
        y_obs = y_true + np.random.randn(m) * obs_noise_slider.value
        
        print(f"\n✓ Données observées générées")
        
        # ====================================================================
        # CALCUL POSTERIOR (analytique pour linéaire)
        # ====================================================================
        
        print("\n⏳ Calcul du posterior...")
        
        post_mean, post_cov = compute_posterior_gaussian(
            y_obs, forward_func, prior_mean, prior_cov, obs_cov
        )
        
        print("✓ Posterior calculé")
        
        # ====================================================================
        # ÉCHANTILLONNAGE
        # ====================================================================
        
        results = {}
        
        if method_toggle.value in ['LIS', 'Both']:
            print("\n⏳ LIS : Échantillonnage du posterior...")
            
            X_post = np.random.multivariate_normal(
                post_mean, post_cov, n_samples_post_slider.value
            )
            
            # Gradients du log-likelihood
            # Pour N(Ax, Σ_obs) : ∇ ln π(y|x) = A^T Σ_obs^{-1} (y - Ax)
            log_like_grads = np.zeros((n_samples_post_slider.value, d))
            
            obs_precision = np.linalg.inv(obs_cov)
            
            for i in range(n_samples_post_slider.value):
                G_i, A_T = forward_func(X_post[i])
                if A_T.ndim == 3:
                    A = A_T[0].T
                else:
                    A = A_T.T
                residual = y_obs - G_i
                log_like_grads[i] = A.T @ obs_precision @ residual
            
            print("✓ Gradients calculés")
            
            print("\n⏳ Construction LIS...")
            
            lis = LikelihoodInformedSubspaces(kl_threshold=kl_threshold_slider.value)
            lis.fit(log_like_grads, prior_cov, data_observed=y_obs)
            
            results['lis'] = lis
            results['X_post'] = X_post
            
            print(f"✓ LIS calculé : {lis.n_components} directions")
        
        if method_toggle.value in ['DA-LIS', 'Both']:
            print("\n⏳ DA-LIS : Échantillonnage du prior...")
            
            X_prior = np.random.multivariate_normal(
                prior_mean, prior_cov, n_samples_prior_slider.value
            )
            
            # Gradients du forward model
            forward_grads = []
            for i in range(n_samples_prior_slider.value):
                _, grad = forward_func(X_prior[i])
                forward_grads.append(grad)
            
            forward_grads = np.array(forward_grads)
            
            if forward_grads.ndim == 2:
                forward_grads = forward_grads[:, :, np.newaxis]
            
            print("✓ Gradients forward calculés")
            
            print("\n⏳ Construction DA-LIS...")
            
            da_lis = DataAveragedLIS(kl_threshold=kl_threshold_slider.value)
            da_lis.fit(forward_grads, prior_cov, obs_cov)
            
            results['da_lis'] = da_lis
            results['X_prior'] = X_prior
            
            print(f"✓ DA-LIS calculé : {da_lis.n_components} directions")
        
        # ====================================================================
        # VISUALISATION
        # ====================================================================
        
        n_rows = 3 if method_toggle.value == 'Both' else 2
        
        fig = make_subplots(
            rows=n_rows, cols=2,
            subplot_titles=(
                'Eigenvalues Comparison' if method_toggle.value == 'Both' else 'Eigenvalues',
                'Data vs Prior Dominance (λ > 1)',
                'Subspace Overlap' if method_toggle.value == 'Both' else 'Informed Directions',
                'Posterior Projection',
                'KL Error Bound',
                'Parameter Recovery'
            ) if method_toggle.value == 'Both' else (
                'Eigenvalues',
                'Data vs Prior Dominance',
                'Informed Directions',
                'Projection'
            ),
            specs=[
                [{'type': 'bar'}, {'type': 'scatter'}],
                [{'type': 'heatmap'} if method_toggle.value == 'Both' else {'type': 'scatter'},
                 {'type': 'scatter'}]
            ] + ([
                [{'type': 'bar'}, {'type': 'scatter'}]
            ] if method_toggle.value == 'Both' else []),
            vertical_spacing=0.15,
            horizontal_spacing=0.15
        )
        
        # SUBPLOT 1 : Eigenvalues
        if method_toggle.value == 'Both':
            n_eigs = min(len(results['lis'].eigenvalues_), 
                        len(results['da_lis'].eigenvalues_))
            x_pos = list(range(1, n_eigs + 1))
            
            fig.add_trace(
                go.Bar(
                    x=x_pos,
                    y=results['lis'].eigenvalues_[:n_eigs],
                    name='LIS',
                    marker_color='steelblue',
                    opacity=0.7
                ),
                row=1, col=1
            )
            
            fig.add_trace(
                go.Bar(
                    x=x_pos,
                    y=results['da_lis'].eigenvalues_[:n_eigs],
                    name='DA-LIS',
                    marker_color='coral',
                    opacity=0.7
                ),
                row=1, col=1
            )
        else:
            model = results.get('lis', results.get('da_lis'))
            fig.add_trace(
                go.Bar(
                    x=list(range(1, len(model.eigenvalues_) + 1)),
                    y=model.eigenvalues_,
                    marker_color='steelblue'
                ),
                row=1, col=1
            )
        
        fig.add_hline(y=1, line_dash='dash', line_color='red',
                      annotation_text='λ=1', row=1, col=1)
        
        fig.update_yaxes(title_text='λᵢ', row=1, col=1)
        fig.update_xaxes(title_text='Index', row=1, col=1)
        
        # SUBPLOT 2 : Data vs Prior dominance
        if 'lis' in results:
            model = results['lis']
            n_show = min(20, len(model.eigenvalues_))
            
            fig.add_trace(
                go.Scatter(
                    x=list(range(1, n_show + 1)),
                    y=model.eigenvalues_[:n_show],
                    mode='markers+lines',
                    marker=dict(size=10),
                    name='LIS'
                ),
                row=1, col=2
            )
            
            fig.add_hrect(y0=1, y1=max(model.eigenvalues_[:n_show].max(), 2),
                         fillcolor='green', opacity=0.1,
                         annotation_text='Data-dominated', row=1, col=2)
            
            fig.add_hrect(y0=0, y1=1,
                         fillcolor='red', opacity=0.1,
                         annotation_text='Prior-dominated', row=1, col=2)
            
            fig.update_yaxes(type='log', title_text='λᵢ (log)', row=1, col=2)
            fig.update_xaxes(title_text='Index', row=1, col=2)
        
        # SUBPLOT 3 : Overlap ou directions
        if method_toggle.value == 'Both':
            r_show = min(results['lis'].n_components, 
                        results['da_lis'].n_components, 5)
            
            overlap = np.abs(
                results['lis'].informed_directions_[:, :r_show].T @ 
                results['da_lis'].informed_directions_[:, :r_show]
            )
            
            fig.add_trace(
                go.Heatmap(
                    z=overlap,
                    x=[f'DA-LIS {i+1}' for i in range(r_show)],
                    y=[f'LIS {i+1}' for i in range(r_show)],
                    colorscale='Blues',
                    zmin=0, zmax=1
                ),
                row=2, col=1
            )
        else:
            # Afficher les premières directions
            model = results.get('lis', results.get('da_lis'))
            n_dir_show = min(3, model.n_components)
            
            for i in range(n_dir_show):
                fig.add_trace(
                    go.Bar(
                        x=list(range(1, d + 1)),
                        y=model.informed_directions_[:, i],
                        name=f'Direction {i+1}'
                    ),
                    row=2, col=1
                )
            
            fig.update_xaxes(title_text='Parameter dimension', row=2, col=1)
            fig.update_yaxes(title_text='Component value', row=2, col=1)
        
        # SUBPLOT 4 : Projection posterior
        if 'lis' in results and 'X_post' in results:
            X_proj = results['lis'].transform(results['X_post'][:500])
            
            if X_proj.shape[1] >= 2:
                fig.add_trace(
                    go.Scatter(
                        x=X_proj[:, 0],
                        y=X_proj[:, 1],
                        mode='markers',
                        marker=dict(
                            size=4,
                            color=np.arange(len(X_proj)),
                            colorscale='Viridis',
                            showscale=True
                        ),
                        name='Posterior'
                    ),
                    row=2, col=2
                )
                
                fig.update_xaxes(title_text='ũ₁ᵀ Σ⁻¹ x', row=2, col=2)
                fig.update_yaxes(title_text='ũ₂ᵀ Σ⁻¹ x', row=2, col=2)
        
        # SUBPLOTS 5-6 (si Both)
        if method_toggle.value == 'Both':
            # SUBPLOT 5 : KL bounds
            kl_lis = results['lis'].estimate_kl_error()
            kl_dalis = results['da_lis'].eigenvalues_[results['da_lis'].n_components:].sum() / 2
            
            fig.add_trace(
                go.Bar(
                    x=['LIS', 'DA-LIS'],
                    y=[kl_lis, kl_dalis],
                    marker_color=['steelblue', 'coral'],
                    text=[f'{kl_lis:.4e}', f'{kl_dalis:.4e}'],
                    textposition='auto'
                ),
                row=3, col=1
            )
            
            fig.update_yaxes(type='log', title_text='D_KL bound', row=3, col=1)
            
            # SUBPLOT 6 : Parameter recovery
            fig.add_trace(
                go.Scatter(
                    x=list(range(1, d + 1)),
                    y=x_true,
                    mode='markers+lines',
                    marker=dict(size=10, color='red'),
                    name='True x'
                ),
                row=3, col=2
            )
            
            fig.add_trace(
                go.Scatter(
                    x=list(range(1, d + 1)),
                    y=post_mean,
                    mode='markers+lines',
                    marker=dict(size=8, color='blue'),
                    name='Posterior mean',
                    error_y=dict(
                        type='data',
                        array=2*np.sqrt(np.diag(post_cov)),
                        visible=True
                    )
                ),
                row=3, col=2
            )
            
            fig.update_xaxes(title_text='Parameter index', row=3, col=2)
            fig.update_yaxes(title_text='Value', row=3, col=2)
        
        # Layout
        fig.update_layout(
            height=900 if method_toggle.value == 'Both' else 700,
            showlegend=True,
            title_text=f'<b>Likelihood-Informed Subspaces</b> ({method_toggle.value})'
        )
        
        fig.show()
        
        # ====================================================================
        # STATISTIQUES
        # ====================================================================
        
        print("\n" + "="*70)
        print("STATISTIQUES")
        print("="*70)
        
        if 'lis' in results:
            print("\nLIS :")
            print(f"  Dimension informée  : {results['lis'].n_components}")
            print(f"  Borne D_KL          : {results['lis'].estimate_kl_error():.4e}")
            print(f"  Directions data-dom : {np.sum(results['lis'].eigenvalues_ > 1)}")
        
        if 'da_lis' in results:
            print("\nDA-LIS :")
            print(f"  Dimension informée  : {results['da_lis'].n_components}")
            kl_da = results['da_lis'].eigenvalues_[results['da_lis'].n_components:].sum() / 2
            print(f"  Borne 𝔼[D_KL]       : {kl_da:.4e}")
            print(f"  Directions data-dom : {np.sum(results['da_lis'].eigenvalues_ > 1)}")
        
        if method_toggle.value == 'Both':
            print("\nComparaison :")
            r_comp = min(results['lis'].n_components, results['da_lis'].n_components)
            overlap_norm = np.linalg.norm(
                results['lis'].informed_directions_[:, :r_comp].T @ 
                results['da_lis'].informed_directions_[:, :r_comp] - np.eye(r_comp),
                'fro'
            )
            print(f"  Distance sous-espaces : {overlap_norm:.4f}")


compute_button_lis.on_click(compute_lis_interactive)

In [7]:
# ============================================================================
# INTERFACE
# ============================================================================

problem_box = widgets.VBox([
    widgets.HTML("<h3>🎯 Inverse Problem Setup</h3>"),
    d_param_slider,
    m_obs_slider,
    forward_model_dropdown,
    obs_noise_slider,
    prior_type_dropdown
])

sampling_box = widgets.VBox([
    widgets.HTML("<h3>📊 Sampling</h3>"),
    n_samples_post_slider,
    n_samples_prior_slider
])

lis_params_box = widgets.VBox([
    widgets.HTML("<h3>🔧 LIS Parameters</h3>"),
    kl_threshold_slider,
    method_toggle,
    compute_button_lis
])

controls_lis = widgets.HBox([problem_box, sampling_box, lis_params_box])

display(widgets.VBox([
    widgets.HTML("<h1>🔍 LIS Interactive Explorer</h1>"),
    widgets.HTML("<p>Explore Likelihood-Informed Subspaces for Bayesian inverse problems</p>"),
    controls_lis,
    output_lis
]))